In [ ]:
import requests
import time
import os
import pandas as pd
from datetime import date
from dotenv import load_dotenv

print("Libraries loaded successfully.")

Libraries loaded successfully!


In [45]:
load_dotenv()

API_KEY = os.getenv("API_KEY")

print("API Key loaded")

API Key loaded


In [58]:
url = "https://api.marketcheck.com/v2/search/car/active"

querystring = {
                "api_key": API_KEY,
                "has_price": "true",
                "price_range": "3000-9000",
                "car_type": "used",
                "vehicle_status": "Available",
                "miles_range": "0-180000",
                "year-range": "1995-2009",
                "rows": 50,
                "start": 0,
}

headers = {"Accept": "application/json"}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'num_found': 9904, 'listings': [{'id': 'KNDJN2A26F7759849-af8429d6-d27c', 'vin': 'KNDJN2A26F7759849', 'heading': '2015 Kia Soul Base FWD Hatchback', 'price': 8577, 'miles': 73588, 'msrp': 8577, 'data_source': 'mc', 'vdp_url': 'https://www.huffines.net/used/Kia/2015-Kia-Soul-096a2ec6ac18019c806aea5d4abe8b98.htm', 'carfax_1_owner': False, 'carfax_clean_title': False, 'exterior_color': 'Alien II', 'interior_color': 'Black', 'base_int_color': 'Black', 'base_ext_color': 'Green', 'dom': 52, 'dom_180': 47, 'dom_active': 47, 'dos_active': 47, 'seller_type': 'dealer', 'inventory_type': 'used', 'stock_no': '26SR1517B', 'last_seen_at': 1778081759, 'last_seen_at_date': '2026-05-06T15:35:59.000Z', 'scraped_at': 1776471220, 'scraped_at_date': '2026-04-18T00:13:40.000Z', 'first_seen_at': 1776471220, 'first_seen_at_date': '2026-04-18T00:13:40.000Z', 'first_seen_at_source': 1774051559, 'first_seen_at_source_date': '2026-03-21T00:05:59.000Z', 'first_seen_at_mc': 1412219648, 'first_seen_at_mc_date': '20

**Pagination Limit**

After trying to loop over with "for start in range (0, 50, 9904)" - because of Market-Check API's stated 50 row per request limit - I discovered a pagination limit of 500 rows

I am going to get around this by splitting the price-range into 3000-6000 and 6001-9000, and then looping over both individual years and each price-range. This will make it so my pagination count resets for each loop, theoretically allowing me to gather 500x14x2 = 14000 listings, way above the total of 9904 found in the last cell. 

In [ ]:

all_listings = []
rows_per_page = 50

years = [str(y) for y in range(1995, 2010)]  # ['1995', '1996', ... '2009']
price_ranges = ["3000-6000", "6001-9000"]

for year in years:
    for price_range in price_ranges:
        print(f"\n--- Fetching year={year}, price={price_range} ---")
        
        for start in range(0, 500, rows_per_page):
            querystring = {
                "api_key": API_KEY,
                "has_price": "true",
                "price_range": price_range,
                "car_type": "used",
                "vehicle_status": "Available",
                "miles_range": "0-180000",
                "year": year,  # single year
                "rows": rows_per_page,
                "start": start,
            }
            
            response = requests.get(url, headers=headers, params=querystring)
            data = response.json()
            listings = data.get("listings", [])
            
            if not listings:
                break
                
            all_listings.extend(listings)
            print(f"Total collected: {len(all_listings)}")
            time.sleep(0.5)


df = pd.DataFrame(all_listings)
df.to_csv("DataSet1-raw_listings.csv", index=False)
print(f"\nDone! Saved {len(df)} unique listings.")

**Concerns of sampling bias:**
When running the 1 - call test in the 2nd code cell I got a total num_found of 9904, meaning 9904 listings satisfy the params. 

I was expecting to get all 9904 because I had a theoretical limit of 14000. However, I only got 2040 listings. 

At first, I thought perhaps the listings were concentrated within certain years and price ranges, however after checking the full response from the cell there was no year and price range that had 500 total collected or more. 

Checking my API usage at MarketCheck.com, I still have 379 calls left

It cannot be the stated 100 mile radious restriction, as looking through raw_listings_csv I have listings from dealerships in Indiana, California, and Nebraska. 

------

Therefore, there is a large concern of sampling bias within raw_listings_csv, as MarketCheck could have organied the API in a specific order, so for the years where I got a limited amount from the total number, I could have only gathered vehicles within the first portion of that order (ex. If by alphabetical, I could have gotten all of the Acuras and Buicks, but no Scion or Volvo cars)

However, looking through the 2009 year (the year with the most gathered listings) in raw_listings CSV, I see listings from Acura to Scion (no alphabetical bias), and both a Honda Fit 1.5L and 4.3L Silvarado (no engine or vehicle size bias).

Thus, it is likely either ...
1. The API is likely organized temporally, favoring dealerships who indexed earlier, 
or 
2. My free-tier API subscription limited me in some way that did not produce an error message when fetching data

**However, checking for bias will still be done . . .**




In [ ]:
df = pd.read_csv("DataSet1-raw_listings.csv")

df_slice = df[df['heading'].str.contains('2009')]
              
print(f"Total 2009 listings: {len(df_slice)}")

df_slice['make'] = df_slice['heading'].str.split().str[1]

import ast

Total 2009 listings: 374


In [68]:
print("\n--- Top Sources ---")
print(df_slice['source'].value_counts().head(10))


--- Top Sources ---
source
econoautosale.com             11
518auto.com                    4
springsautomotivegroup.com     4
eaglevalleymotors.com          4
corwinwholesale.com            3
dmmotorswalledlake.com         3
sportscarsintl.com             3
shopi-car.com                  3
axioev.com                     3
raautosalesca.com              3
Name: count, dtype: int64


In [69]:
print("\n--- Make Distribution ---")
print(df_slice['make'].value_counts())


--- Make Distribution ---
make
Ford             44
Toyota           43
Chevrolet        32
Nissan           25
Honda            23
2009             19
Hyundai          18
Volkswagen       14
Subaru           12
MINI             11
Dodge            10
BMW              10
Mazda             8
Jeep              8
Kia               8
FORD              7
CADILLAC          6
LEXUS             6
Pontiac           5
Scion             5
Buick             5
HONDA             5
Chrysler          4
INFINITI          4
Mercury           4
GMC               4
Lincoln           4
TOYOTA            4
Audi              3
KIA               3
Acura             3
Mercedes-Benz     3
Jaguar            2
CHEVROLET         2
VOLKSWAGEN        2
Mitsubishi        1
Volvo             1
NISSAN            1
HYUNDAI           1
|                 1
SUBARU            1
PONTIAC           1
SCION             1
Name: count, dtype: int64


_↑ Warning: Car make data must be normalized ↑_ 

In [70]:
print("\n--- Price Distribution ---")
print(df_slice['price'].describe())


--- Price Distribution ---
count     374.000000
mean     6941.056150
std      1451.929903
min      3075.000000
25%      5995.000000
50%      6998.000000
75%      7995.000000
max      9000.000000
Name: price, dtype: float64


In [71]:
print("\n--- Miles Distribution ---")
print(df_slice['miles'].describe())


--- Miles Distribution ---
count       374.000000
mean     126091.462567
std       34466.704347
min       11565.000000
25%      102309.250000
50%      128947.000000
75%      153978.250000
max      179871.000000
Name: miles, dtype: float64


In [72]:
print("\n--- Days on Market ---")
print(df_slice['dom'].describe())


--- Days on Market ---
count     374.000000
mean      117.120321
std       228.171715
min         1.000000
25%        13.000000
50%        38.000000
75%       100.750000
max      1687.000000
Name: dom, dtype: float64


In [ ]:
df_slice['state'] = df_slice['dealer'].apply(
    lambda x: ast.literal_eval(x).get('state') if pd.notna(x) else None
)
print("\n--- State Distribution ---")
print(df_slice['state'].value_counts())

**Post-check analysis:**    
All checks came through clean. All that I am looking for this analysis is the existance of minimum and maximum values that either match those given in the params (price & mileage), or distributions that are not unreasonably skewed. 

For example, state and make are reasonably biased distributions, as data is likely to be biased towards more populous states (other than NY wierdly) and more popular brands. 

The one interesting point of data captured was the 75th quartile of days on market being 100 days, while the maximum was 1687 days (4.5 years!)

In [74]:
df[df['dom'] == df['dom'].max()]

,id,vin,heading,price,miles,msrp,data_source,vdp_url,carfax_1_owner,carfax_clean_title,...,source,in_transit,vehicle_status,availability_status,media,dealer,mc_dealership,build,model_code,is_certified
133,2FMZA5148YBA19470-b511c8d6-ffdf,2FMZA5148YBA19470,2000 Ford Windstar Wagon LX,7830,55000,7830,mc,https://www.rolandrichforddelavan.com/used-Del...,False,False,...,rolandrichforddelavan.com,False,Available,NaN,{'photo_links': ['https://www.rolandrichfordde...,"{'id': 1008600, 'website': 'rolandrichforddela...","{'mc_website_id': 1008600, 'mc_dealer_id': 110...","{'year': 2000, 'make': 'Ford', 'model': 'Winds...",A51,NaN


**↑ 4.5 years on market listing data ↑**     
Autotrader says that after 90 days most cars on the lot are sold to wholesale auctions. The mileage isn't excessive for the price either. The title isn't clean, although this is common within the data.     Perhaps this is an error in the data?